# Notebook 8: Feature Scaling

Learn feature scaling with small, self-contained examples. Run cells from top to bottom.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, Normalizer

df = pd.DataFrame({"Age": [22, 28, 35, 42, 50], "Salary": [20000, 30000, 45000, 60000, 90000]})
print(df)

   Age  Salary
0   22   20000
1   28   30000
2   35   45000
3   42   60000
4   50   90000


## 1. Why Feature Scaling?

### Definition
Feature scaling changes the numerical range of input features without changing which records they describe. Features measured in different units can have very different magnitudes. Large values may dominate distance calculations or optimization. Scaling makes comparisons more balanced.

### Business Example
A bank records customer age in years and annual income in rupees. Income values are much larger than age values. Without scaling, a distance-based customer grouping may focus mostly on income. Scaling gives both features a more comparable influence.

### AI/ML Use
KNN and K-Means compare distances between samples. Gradient-based models may also train more smoothly when features are similarly scaled. Scaling should be fitted using training data only. It does not automatically improve every model.

### Python Example

In [3]:
print("Before scaling:\n", df)
print("After scaling:\n", pd.DataFrame(StandardScaler().fit_transform(df), columns=df.columns).round(2))

Before scaling:
    Age  Salary
0   22   20000
1   28   30000
2   35   45000
3   42   60000
4   50   90000
After scaling:
     Age  Salary
0 -1.35   -1.18
1 -0.75   -0.77
2 -0.04   -0.16
3  0.67    0.45
4  1.47    1.67


### Explanation
The code displays age and salary before and after standardization. Both columns become centered near zero with comparable spread. The original rows are preserved. In a real project, fit the scaler only on the training split.

### When to Use / When Not to Use
Use when features have very different scales and the algorithm is scale-sensitive. Avoid assuming scaling fixes missing values or incorrect records.

## 2. Scale-Sensitive Algorithms

### Definition
Scale-sensitive algorithms change their behavior when feature magnitudes change. They commonly rely on distances, dot products, or gradient optimization. A large-scale feature can overpower a small-scale feature. Scaling often helps these methods behave as intended.

### Business Example
A retailer groups customers using age and total spending. Spending values may be thousands while ages are only tens. K-Means can place too much emphasis on spending without scaling. Standardizing both inputs creates more balanced clusters.

### AI/ML Use
KNN, K-Means, SVM with RBF kernels, PCA, and regularized linear models often benefit from scaling. Distance and penalty calculations depend on numerical magnitude. Fit preprocessing on training data before evaluating. Test the impact rather than assuming improvement.

### Python Example

In [5]:
from sklearn.metrics.pairwise import euclidean_distances
print("Original distances:\n", euclidean_distances(df.iloc[:3]).round(1))
print("Scaled distances:\n", euclidean_distances(StandardScaler().fit_transform(df.iloc[:3])).round(2))

Original distances:
 [[    0. 10000. 25000.]
 [10000.     0. 15000.]
 [25000. 15000.     0.]]
Scaled distances:
 [[0.   1.49 3.45]
 [1.49 0.   1.97]
 [3.45 1.97 0.  ]]


### Explanation
Euclidean distance measures how far rows are from each other. Before scaling, salary contributes much more to distance. After scaling, age and salary both influence the result. This explains why distance-based algorithms can be scale-sensitive.

### When to Use / When Not to Use
Use scaling for distance-based methods and many gradient-based models. Avoid comparing model performance without consistent preprocessing.

## 3. Scale-Insensitive Algorithms

### Definition
Scale-insensitive algorithms usually depend on feature order or split thresholds instead of distance magnitude. Multiplying a feature by a positive constant does not normally change its ranking. Decision trees can split on either the original or rescaled threshold. Scaling is therefore usually unnecessary for tree-based models.

### Business Example
A company predicts customer churn using age and account balance. A decision tree searches for useful split points. Whether balance is stored in rupees or thousands of rupees, the ordering stays the same. Its decisions generally remain equivalent.

### AI/ML Use
Decision Trees, Random Forests, and many gradient-boosted tree models usually do not need scaling. They select threshold-based rules rather than distance measures. Other preprocessing may still be needed. Scaling can still be used when combining trees with other algorithms.

### Python Example

In [7]:
from sklearn.tree import DecisionTreeRegressor
y = [1, 2, 3, 4, 5]
a = DecisionTreeRegressor(max_depth=2, random_state=0).fit(df, y)
b = DecisionTreeRegressor(max_depth=2, random_state=0).fit(StandardScaler().fit_transform(df), y)
print("Same predictions:", np.allclose(a.predict(df), b.predict(StandardScaler().fit_transform(df))))

Same predictions: True


### Explanation
The code trains two small trees with original and standardized features. Their predictions match on the example records. Tree thresholds can change numerically while producing equivalent partitions. This is why scaling is not usually required for decision trees.

### When to Use / When Not to Use
Use tree models without scaling unless a broader pipeline needs it. Do not assume every algorithm is scale-insensitive.

## 4. Normalization

### Definition
Normalization can mean converting each sample vector to unit length. Unlike column scaling, row normalization adjusts all features within a single record together. It emphasizes direction rather than overall magnitude. This meaning differs from the common use of “normalization” for min-max scaling.

### Business Example
A document may contain word counts of very different total lengths. Unit-length normalization can compare word patterns rather than document size. A longer document no longer automatically has a larger vector length. This can help text similarity calculations.

### AI/ML Use
Cosine-similarity workflows and text feature vectors often use row normalization. It can be useful when relative proportions matter more than magnitude. It may discard useful size information. Choose it based on the task rather than as a default.

### Python Example

In [9]:
sample = np.array([[3, 4], [6, 8]])
print(Normalizer().fit_transform(sample))

[[0.6 0.8]
 [0.6 0.8]]


### Explanation
Both input rows point in the same direction. The Normalizer changes each row to unit Euclidean length. Both become [0.6, 0.8]. This demonstrates row normalization, not per-column min-max scaling.

### When to Use / When Not to Use
Use for direction-based comparisons such as text vectors. Avoid when absolute magnitude is important.

## 5. Standardization

### Definition
Standardization subtracts the feature mean and divides by its standard deviation. The resulting training feature usually has mean near zero and standard deviation near one. It does not force values into a fixed interval. Extreme values can strongly affect the mean and standard deviation.

### Business Example
Employee salaries vary widely across a company. Standardization expresses salaries relative to the average and typical variation. A positive standardized value indicates an above-average salary. Very high salaries may still produce large standardized values.

### AI/ML Use
Standardization is a common choice for SVM, PCA, logistic regression, and KNN. It makes numerical scales more comparable. It is not resistant to strong outliers. Use training-set statistics when transforming validation and test data.

### Python Example

In [11]:
values = pd.DataFrame({"Salary": [20000, 30000, 45000, 60000]})
print(pd.DataFrame(StandardScaler().fit_transform(values), columns=["Salary"]).round(2))

   Salary
0   -1.24
1   -0.58
2    0.41
3    1.40


### Explanation
StandardScaler calculates the mean and standard deviation of the column. It then converts each salary to a standardized score. Negative results are below the mean and positive results are above it. The relative ordering of salaries stays unchanged.

### When to Use / When Not to Use
Use for roughly well-behaved numeric features in scale-sensitive models. Avoid relying on it to reduce the impact of extreme outliers.

## 6. Min-Max Scaling

### Definition
Min-max scaling maps a feature into a chosen interval, commonly zero to one. It uses the minimum and maximum values observed during fitting. The smallest training value maps to zero and the largest to one. New values outside the training range can transform outside zero to one.

### Business Example
A shopping application records ratings from 1 to 5 and spending in rupees. Min-max scaling can place both numerical columns on a common interval. This makes their magnitudes easier to compare. One unusually large spending value can compress the other values near zero.

### AI/ML Use
Min-max scaling is useful when bounded inputs are helpful. It is sometimes used in neural-network workflows and distance-based models. It is sensitive to extreme minimum and maximum values. Fit it only on training data to avoid leakage.

### Python Example

In [13]:
values = pd.DataFrame({"Score": [20, 40, 60, 80]})
print(MinMaxScaler().fit_transform(values))

[[0.        ]
 [0.33333333]
 [0.66666667]
 [1.        ]]


### Explanation
The minimum score becomes 0 and the maximum becomes 1. The middle values are placed proportionally between them. The order and relative gaps are preserved. A new score above 80 would scale above 1 unless clipping is enabled.

### When to Use / When Not to Use
Use when a fixed training-data interval is helpful. Avoid as the first choice when extreme values distort the range.

## 7. StandardScaler

### Definition
StandardScaler is a scikit-learn tool that standardizes columns. It learns a mean and standard deviation from the fitted data. The transform method applies the learned statistics to other rows. This makes it suitable for reproducible machine-learning pipelines.

### Business Example
A loan model uses age and income as numerical inputs. StandardScaler learns the average and spread from historical training customers. New applications are transformed using those same values. Recalculating statistics on each new application would create inconsistent inputs.

### AI/ML Use
StandardScaler works well in pipelines with logistic regression, SVM, and PCA. It should be fitted only on the training subset. Validation and test records use transform rather than fit_transform. Outliers can still influence its learned statistics.

### Python Example

In [15]:
scaler = StandardScaler()
scaled = scaler.fit_transform(df)
print(pd.DataFrame(scaled, columns=df.columns).round(2))

    Age  Salary
0 -1.35   -1.18
1 -0.75   -0.77
2 -0.04   -0.16
3  0.67    0.45
4  1.47    1.67


### Explanation
The scaler fits separately to the Age and Salary columns. It stores each column’s training mean and spread. The printed values are standardized versions of the original numbers. In evaluation, use scaler.transform on unseen data.

### When to Use / When Not to Use
Use with many scale-sensitive algorithms. Avoid fitting on all data before splitting into training and test sets.

## 8. MinMaxScaler

### Definition
MinMaxScaler is a scikit-learn tool for min-max scaling. By default it maps the training minimum to zero and maximum to one. It can also map data to a different chosen interval. Extreme values can strongly influence the mapping.

### Business Example
A service measures daily usage in minutes. MinMaxScaler can express usage on a zero-to-one training scale. Low usage becomes close to zero and high usage close to one. A single extreme day can compress ordinary days into a narrow interval.

### AI/ML Use
MinMaxScaler can be helpful when bounded numerical inputs suit a model. It preserves the order of observations. It does not reduce skewness or remove outliers. New observations can exceed the training range after transformation.

### Python Example

In [17]:
scaler = MinMaxScaler()
print(pd.DataFrame(scaler.fit_transform(df), columns=df.columns).round(2))

    Age  Salary
0  0.00    0.00
1  0.21    0.14
2  0.46    0.36
3  0.71    0.57
4  1.00    1.00


### Explanation
The scaler finds the minimum and maximum for each column. It maps the observed training range to zero through one. Age and Salary are transformed independently. The original feature relationships within each column remain ordered.

### When to Use / When Not to Use
Use for features where a bounded training range is useful. Avoid assuming it is robust to outliers.

## 9. RobustScaler

### Definition
RobustScaler centers features using the median and scales them using the interquartile range. The median and IQR are less affected by extreme values than the mean and standard deviation. This can preserve useful variation among typical observations. The output is not restricted to a fixed range.

### Business Example
Most customers spend between ₹500 and ₹2,000, but one spends ₹1,00,000. Standard scaling can be heavily influenced by that large purchase. RobustScaler focuses on the median and middle half of spending values. The unusual customer remains present in the dataset.

### AI/ML Use
RobustScaler is useful for scale-sensitive models when features contain genuine outliers. It does not detect or delete errors. Extreme transformed values may still remain large. Review whether the outliers are valid before choosing a treatment.

### Python Example

In [19]:
spend = pd.DataFrame({"Spend": [500, 700, 900, 1100, 100000]})
print(RobustScaler().fit_transform(spend).round(2))

[[ -1.  ]
 [ -0.5 ]
 [  0.  ]
 [  0.5 ]
 [247.75]]


### Explanation
The code scales spending with median and IQR. The very large purchase remains in the data rather than being removed. Typical purchases are scaled relative to the middle of the distribution. This makes the scaler less sensitive to extreme values.

### When to Use / When Not to Use
Use when genuine extreme values are present. Avoid assuming it makes all extreme transformed values small.

## 10. MaxAbsScaler

### Definition
MaxAbsScaler divides each feature by its largest absolute training value. It usually maps training values into the interval minus one to one. It preserves zero values, which is useful for sparse datasets. Like min-max scaling, it is sensitive to extreme values.

### Business Example
A text classification dataset contains many zero word-count entries. MaxAbsScaler leaves zeros unchanged while rescaling nonzero counts. This helps retain a sparse representation. A single very large count can still influence the scale.

### AI/ML Use
MaxAbsScaler is often useful with sparse matrices and features already centered near zero. It does not subtract a mean, so zeros remain zero. It can support models that benefit from comparable scales. It is not robust to extreme maximum values.

### Python Example

In [21]:
values = pd.DataFrame({"Change": [-20, 0, 10, 40]})
print(MaxAbsScaler().fit_transform(values))

[[-0.5 ]
 [ 0.  ]
 [ 0.25]
 [ 1.  ]]


### Explanation
The maximum absolute value is 40. Every number is divided by 40, giving values from -0.5 to 1. Zero remains exactly zero. This is especially helpful when maintaining sparsity matters.

### When to Use / When Not to Use
Use for sparse or zero-heavy numeric data. Avoid relying on it when the maximum absolute value is an outlier.